In [1]:

import json
import re
import os
import shutil
from pathlib import Path

import torch
import cupy as cp
import numpy as np

# ==========================================
# DLPACK COMPATIBILITY WORKAROUND
# MUST RUN BEFORE IMPORTING SPACY
# ==========================================

import thinc.util as thinc_util


def xp2torch_compat(xp_tensor, requires_grad=False, device=None):

    if device is None:
        device = thinc_util.get_torch_default_device()

    if isinstance(xp_tensor, cp.ndarray):

        # Avoid DLPack capsule errors
        numpy_array = cp.asnumpy(xp_tensor)
        torch_tensor = torch.from_numpy(numpy_array)

    elif isinstance(xp_tensor, np.ndarray):

        torch_tensor = torch.from_numpy(xp_tensor)

    elif isinstance(xp_tensor, torch.Tensor):

        torch_tensor = xp_tensor

    else:
        raise TypeError(
            f"Unsupported input type: {type(xp_tensor)}"
        )

    torch_tensor = torch_tensor.to(device)

    if requires_grad:
        torch_tensor.requires_grad_()

    return torch_tensor


# Apply patch BEFORE importing spaCy
thinc_util.xp2torch = xp2torch_compat


# ==========================================
# IMPORT SPACY AFTER PATCH
# ==========================================

import spacy
import spacy_curated_transformers


# ==========================================
# GPU CONFIGURATION
# ==========================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available!")

spacy.require_gpu()

nlp = spacy.load("en_core_web_trf")

print("GPU:", torch.cuda.get_device_name(0))
print("spaCy GPU configuration successful!")


# ==========================================
# CONFIGURATION
# ==========================================

INPUT_FILE = "./dataset/V2D/V2D.jsonl"

OUTPUT_FILE = "./dataset/V2D/V2Dclean.jsonl"
REMOVED_FILE = "./dataset/V2D/V2Dremoved.jsonl"

BACKUP_FILE = "./dataset/V2D/V2D_backup.jsonl"

FIELDS_TO_CHECK = [
    "conversation",
    "conversation_summary"
]

# Processing configuration
BATCH_SIZE = 8
CHUNK_SIZE = 32

# Automatically replace original dataset
REPLACE_ORIGINAL = True


# ==========================================
# ROLE WORDS TO IGNORE
# ==========================================

ROLE_WORDS = {
    # Conversation roles
    "user", "assistant", "system", "developer", "admin",
    "administrator", "agent", "bot", "support",
    "moderator", "operator",

    # Relationships and family
    "friend", "partner", "mentor", "roommate", "roommates",
    "mother", "father", "mom", "dad", "mama", "papa",
    "brother", "sister", "sibling", "siblings",
    "son", "daughter", "child", "children",
    "parent", "parents", "grandma", "grandpa",
    "grandmother", "grandfather", "grandparents",
    "uncle", "aunt", "cousin", "cousins",
    "husband", "wife", "spouse", "fiance", "fiancee",
    "boyfriend", "girlfriend", "relative", "relatives",
    "family", "family member", "in-law",

    # Education
    "teacher", "student", "professor", "lecturer",
    "tutor", "instructor", "principal", "trainer",
    "educator", "classmate", "classmates", "headmaster",

    # Medical and healthcare
    "doctor", "nurse", "therapist", "psychologist",
    "psychiatrist", "counselor", "surgeon", "dentist",
    "physician", "specialist", "pharmacist", "patient",
    "caregiver", "paramedic",

    # Professional roles
    "manager", "employee", "employer", "boss",
    "colleague", "coworker", "team", "teammate",
    "customer", "client", "consultant", "director",
    "executive", "supervisor", "assistant manager",
    "secretary", "receptionist", "accountant",
    "engineer", "developer", "designer", "programmer",
    "lawyer", "attorney", "judge", "police",
    "officer", "soldier", "chef", "waiter", "waitress",
    "driver", "salesperson", "entrepreneur",
    "business owner", "founder", "ceo", "cto",
    "intern", "freelancer", "worker", "staff",

    # Coaching and guidance
    "coach", "career coach", "life coach",
    "fitness coach", "business coach", "advisor",
    "adviser", "guide", "life advisor", "career advisor",

    # General descriptions
    "person", "people", "individual", "someone",
    "somebody", "stranger", "visitor", "guest",
    "member", "members", "volunteer", "candidate",
    "applicant", "leader", "follower", "owner",
    "customer service", "representative"
}

ROLE_WORDS = {
    word.casefold() for word in ROLE_WORDS
}


# ==========================================
# NAME INTRODUCTION PATTERNS
# ==========================================

NAME_PATTERNS = [
    r"\bmy name is\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})",
    r"\bcall me\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})",
    r"\bdear\s+([A-Z][a-z]+)\b",
    r"\bregards,\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})"
]


# ==========================================
# ROLE WORD CHECK
# ==========================================

def contains_role_word(name):

    words = re.findall(
        r"[a-z]+",
        name.casefold()
    )

    return any(
        word in ROLE_WORDS
        for word in words
    )


# ==========================================
# DETECT PERSONAL NAMES
# ==========================================

def detect_person_names(text, doc):

    if not isinstance(text, str) or not text.strip():
        return []

    detections = []
    seen = set()

    # --------------------------------------
    # 1. SPACY NER DETECTION
    # --------------------------------------

    for ent in doc.ents:

        if ent.label_ not in ("PERSON", "PER"):
            continue

        name = ent.text.strip()

        if contains_role_word(name):
            continue

        key = name.casefold()

        if key not in seen:

            seen.add(key)

            detections.append({
                "name": name,
                "method": "spaCy NER",
                "entity_label": ent.label_
            })

    # --------------------------------------
    # 2. REGEX DETECTION
    # --------------------------------------

    for pattern in NAME_PATTERNS:

        matches = re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:

            name = match.group(1).strip()

            if contains_role_word(name):
                continue

            key = name.casefold()

            if key not in seen:

                seen.add(key)

                detections.append({
                    "name": name,
                    "method": "Regex",
                    "matched_pattern": pattern,
                    "matched_text": match.group(0)
                })

    return detections


# ==========================================
# PROCESS ONE BATCH OF RECORDS
# ==========================================

def process_batch(batch, outfile, removedfile, reason_counts):

    texts = []
    text_metadata = []

    record_reasons = [[] for _ in batch]

    # --------------------------------------
    # 1. COLLECT TEXTS
    # --------------------------------------

    for record_index, (line_number, record) in enumerate(batch):

        input_data = record.get("input", {})

        if not isinstance(input_data, dict):
            continue

        for field in FIELDS_TO_CHECK:

            text = input_data.get(field, "")

            if isinstance(text, str) and text.strip():

                texts.append(text)

                text_metadata.append({
                    "record_index": record_index,
                    "field": field
                })

    # --------------------------------------
    # 2. GPU BATCH PROCESSING
    # --------------------------------------

    if texts:

        docs = nlp.pipe(
            texts,
            batch_size=BATCH_SIZE
        )

        for text_index, (doc, metadata) in enumerate(
            zip(docs, text_metadata)
        ):

            record_index = metadata["record_index"]
            field = metadata["field"]

            original_text = texts[text_index]

            detections = detect_person_names(
                original_text,
                doc
            )

            for detection in detections:

                record_reasons[record_index].append({

                    "field": field,

                    "detected_name": detection["name"],

                    "detection_method": detection["method"],

                    "details": detection
                })

    # --------------------------------------
    # 3. SAVE RECORDS
    # --------------------------------------

    kept = 0
    removed = 0

    for index, (line_number, record) in enumerate(batch):

        reasons = record_reasons[index]

        if reasons:

            record["_removal_reasons"] = reasons
            record["_removed_at_line"] = line_number

            removedfile.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            removed += 1

            for reason in reasons:

                name = reason["detected_name"]

                reason_counts[name] = (
                    reason_counts.get(name, 0) + 1
                )

        else:

            outfile.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            kept += 1

    return kept, removed


# ==========================================
# CLEAN DATASET
# ==========================================

def clean_dataset():

    kept = 0
    removed = 0
    invalid = 0

    reason_counts = {}

    batch = []

    input_path = Path(INPUT_FILE)
    output_path = Path(OUTPUT_FILE)
    removed_path = Path(REMOVED_FILE)
    backup_path = Path(BACKUP_FILE)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_path}"
        )

    if REPLACE_ORIGINAL and backup_path.exists():
        raise FileExistsError(
            f"Backup already exists: {backup_path}\n"
            "Rename or move the existing backup before "
            "running replacement again."
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------
    # OPEN FILES
    # --------------------------------------

    with (
        open(input_path, "r", encoding="utf-8") as infile,
        open(output_path, "w", encoding="utf-8") as outfile,
        open(removed_path, "w", encoding="utf-8") as removedfile
    ):

        for line_number, line in enumerate(infile, start=1):

            if not line.strip():
                continue

            try:

                record = json.loads(line)

                if not isinstance(record, dict):

                    invalid += 1

                    print(
                        f"Invalid record structure at line {line_number}"
                    )

                    continue

                batch.append((line_number, record))

                if len(batch) >= CHUNK_SIZE:

                    batch_kept, batch_removed = process_batch(
                        batch,
                        outfile,
                        removedfile,
                        reason_counts
                    )

                    kept += batch_kept
                    removed += batch_removed

                    batch.clear()

                    processed = kept + removed

                    print(
                        f"Processed: {processed} | "
                        f"Kept: {kept} | "
                        f"Removed: {removed}"
                    )

            except json.JSONDecodeError:

                invalid += 1

                print(
                    f"Invalid JSON at line {line_number}"
                )

        # Process remaining records
        if batch:

            batch_kept, batch_removed = process_batch(
                batch,
                outfile,
                removedfile,
                reason_counts
            )

            kept += batch_kept
            removed += batch_removed

    # ======================================
    # FINAL REPORT
    # ======================================

    print("\n========== DATASET CLEANING ==========")

    print(f"Records processed: {kept + removed}")
    print(f"Records retained: {kept}")
    print(f"Records removed: {removed}")
    print(f"Invalid JSON lines: {invalid}")

    print("\n========== DETECTED PERSONAL NAMES ==========")

    for name, count in sorted(
        reason_counts.items(),
        key=lambda x: x[1],
        reverse=True
    ):

        print(f"{name}: {count} detections")

    print("\n========== OUTPUT FILES ==========")

    print(f"Clean dataset: {output_path}")
    print(f"Removed records: {removed_path}")

    # ======================================
    # REPLACE ORIGINAL DATASET
    # ======================================

    if REPLACE_ORIGINAL:

        # Preserve original dataset
        shutil.copy2(
            input_path,
            backup_path
        )

        # Replace original with cleaned dataset
        os.replace(
            output_path,
            input_path
        )

        print("\n========== ORIGINAL DATASET UPDATED ==========")

        print(f"Updated dataset: {input_path}")
        print(f"Original backup: {backup_path}")

    # ======================================
    # GPU MEMORY REPORT
    # ======================================

    print("\n========== GPU MEMORY ==========")

    print(
        f"GPU memory allocated: "
        f"{torch.cuda.memory_allocated(0) / 1024**3:.2f} GB"
    )

    print(
        f"GPU memory reserved: "
        f"{torch.cuda.memory_reserved(0) / 1024**3:.2f} GB"
    )


# ==========================================
# MAIN
# ==========================================

if __name__ == "__main__":
    clean_dataset()


g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 3060
spaCy GPU configuration successful!
Processed: 32 | Kept: 23 | Removed: 9
Processed: 64 | Kept: 46 | Removed: 18
Processed: 96 | Kept: 65 | Removed: 31
Processed: 128 | Kept: 92 | Removed: 36
Processed: 160 | Kept: 124 | Removed: 36
Processed: 192 | Kept: 154 | Removed: 38
Processed: 224 | Kept: 179 | Removed: 45
Processed: 256 | Kept: 207 | Removed: 49
Processed: 288 | Kept: 232 | Removed: 56
Processed: 320 | Kept: 258 | Removed: 62
Processed: 352 | Kept: 284 | Removed: 68
Processed: 384 | Kept: 307 | Removed: 77
Processed: 416 | Kept: 330 | Removed: 86
Processed: 448 | Kept: 357 | Removed: 91
Processed: 480 | Kept: 387 | Removed: 93
Processed: 512 | Kept: 416 | Removed: 96
Processed: 544 | Kept: 448 | Removed: 96
Processed: 576 | Kept: 475 | Removed: 101
Processed: 608 | Kept: 502 | Removed: 106
Processed: 640 | Kept: 530 | Removed: 110
Processed: 672 | Kept: 544 | Removed: 128
Processed: 704 | Kept: 574 | Removed: 130
Processed: 736 | Kept: 598 | Removed